# Teil 4: Evaluation - Airbnb NYC 2019

In diesem Notizbuch evaluiere ich das Modell aus Teil 3 anhand aussagekräftiger Felder, geeigneter Metriken und einer Wahrheitsmatrix.

## 4.1 Aussagekräftige Felder

Ich bestimme die Wichtigkeit der Felder mit den Feature-Importances des Random-Forest-Modells. Bei kategorialen Feldern werden die One-Hot-Teilausprägungen wieder auf das ursprüngliche Feld aggregiert.

In [5]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv('AB_NYC_2019.csv')
df = df[df['price'] > 0].copy()

feature_spalten = [
    'neighbourhood_group',
    'room_type',
    'latitude',
    'longitude',
    'minimum_nights',
    'number_of_reviews',
    'reviews_per_month',
    'calculated_host_listings_count',
    'availability_365'
]
target_spalte = 'price'

X = df[feature_spalten]
y = df[target_spalte]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numerische_spalten = [
    'latitude',
    'longitude',
    'minimum_nights',
    'number_of_reviews',
    'reviews_per_month',
    'calculated_host_listings_count',
    'availability_365'
]
kategorische_spalten = ['neighbourhood_group', 'room_type']

numerische_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

kategorische_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

vorverarbeitung = ColumnTransformer(
    transformers=[
        ('num', numerische_pipeline, numerische_spalten),
        ('cat', kategorische_pipeline, kategorische_spalten)
    ]
)

modell = Pipeline(steps=[
    ('vorverarbeitung', vorverarbeitung),
    ('regressor', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])

modell.fit(X_train, y_train)
y_vorhersage = modell.predict(X_test)

feature_namen = modell.named_steps['vorverarbeitung'].get_feature_names_out()
importances = modell.named_steps['regressor'].feature_importances_

details = pd.DataFrame({
    'feature': feature_namen,
    'importance': importances
}).sort_values('importance', ascending=False)

def basis_feature_name(name):
    cleaned = name.replace('num__', '').replace('cat__', '')
    for cat in kategorische_spalten:
        prefix = f'{cat}_'
        if cleaned.startswith(prefix):
            return cat
    return cleaned

details['basis_feature'] = details['feature'].apply(basis_feature_name)
aggregiert = details.groupby('basis_feature', as_index=False)['importance'].sum()
aggregiert = aggregiert.sort_values('importance', ascending=False)

print('Top 10 kodierte Features:')
print(details.head(10).to_string(index=False))

print('\nWichtigste ursprüngliche Felder (aggregiert):')
print(aggregiert.to_string(index=False))

Top 10 kodierte Features:
                            feature  importance                  basis_feature
                     num__longitude    0.280883                      longitude
                      num__latitude    0.208978                       latitude
                num__minimum_nights    0.120492                 minimum_nights
              num__availability_365    0.096677               availability_365
             num__reviews_per_month    0.084784              reviews_per_month
     cat__room_type_Entire home/apt    0.067579                      room_type
num__calculated_host_listings_count    0.066442 calculated_host_listings_count
             num__number_of_reviews    0.048241              number_of_reviews
    cat__neighbourhood_group_Queens    0.011766            neighbourhood_group
 cat__neighbourhood_group_Manhattan    0.006688            neighbourhood_group

Wichtigste ursprüngliche Felder (aggregiert):
                 basis_feature  importance
               

## 4.2 Messmetrik

Als Hauptmetrik verwende ich MAE (Mean Absolute Error), weil sie den durchschnittlichen absoluten Fehler direkt in USD angibt und dadurch leicht interpretierbar ist. Zusätzlich berechne ich RMSE und R2, um grössere Fehler stärker zu gewichten und die erklärte Varianz zu beurteilen.

In [6]:
# Diese Zelle funktioniert auch allein nach einem Kernel-Neustart.
if not all(name in globals() for name in ['y_test', 'y_vorhersage']):
    import pandas as pd
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.impute import SimpleImputer
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder

    df = pd.read_csv('AB_NYC_2019.csv')
    df = df[df['price'] > 0].copy()

    feature_spalten = [
        'neighbourhood_group',
        'room_type',
        'latitude',
        'longitude',
        'minimum_nights',
        'number_of_reviews',
        'reviews_per_month',
        'calculated_host_listings_count',
        'availability_365'
    ]
    target_spalte = 'price'

    X = df[feature_spalten]
    y = df[target_spalte]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    numerische_spalten = [
        'latitude',
        'longitude',
        'minimum_nights',
        'number_of_reviews',
        'reviews_per_month',
        'calculated_host_listings_count',
        'availability_365'
    ]
    kategorische_spalten = ['neighbourhood_group', 'room_type']

    numerische_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median'))
    ])

    kategorische_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    vorverarbeitung = ColumnTransformer(
        transformers=[
            ('num', numerische_pipeline, numerische_spalten),
            ('cat', kategorische_pipeline, kategorische_spalten)
        ]
    )

    modell = Pipeline(steps=[
        ('vorverarbeitung', vorverarbeitung),
        ('regressor', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
    ])

    modell.fit(X_train, y_train)
    y_vorhersage = modell.predict(X_test)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_vorhersage)
rmse = mean_squared_error(y_test, y_vorhersage) ** 0.5
r2 = r2_score(y_test, y_vorhersage)

print(f'MAE  (Hauptmetrik): {mae:.2f} USD')
print(f'RMSE:              {rmse:.2f} USD')
print(f'R2:                {r2:.3f}')

MAE  (Hauptmetrik): 65.58 USD
RMSE:              231.78 USD
R2:                0.112


## 4.3 Wahrheitsmatrix, Sensitivität und Spezifizität

Da das Modell Preise kontinuierlich vorhersagt, definiere ich für die Klassifikation eine Bedingung: "teuer" falls Preis >= 200 USD, sonst "nicht teuer". Auf dieser Basis berechne ich die Wahrheitsmatrix sowie Sensitivität und Spezifizität.

In [7]:
# Diese Zelle funktioniert auch allein nach einem Kernel-Neustart.
if not all(name in globals() for name in ['y_test', 'y_vorhersage']):
    import pandas as pd
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.impute import SimpleImputer
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder

    df = pd.read_csv('AB_NYC_2019.csv')
    df = df[df['price'] > 0].copy()

    feature_spalten = [
        'neighbourhood_group',
        'room_type',
        'latitude',
        'longitude',
        'minimum_nights',
        'number_of_reviews',
        'reviews_per_month',
        'calculated_host_listings_count',
        'availability_365'
    ]
    target_spalte = 'price'

    X = df[feature_spalten]
    y = df[target_spalte]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    numerische_spalten = [
        'latitude',
        'longitude',
        'minimum_nights',
        'number_of_reviews',
        'reviews_per_month',
        'calculated_host_listings_count',
        'availability_365'
    ]
    kategorische_spalten = ['neighbourhood_group', 'room_type']

    numerische_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median'))
    ])

    kategorische_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    vorverarbeitung = ColumnTransformer(
        transformers=[
            ('num', numerische_pipeline, numerische_spalten),
            ('cat', kategorische_pipeline, kategorische_spalten)
        ]
    )

    modell = Pipeline(steps=[
        ('vorverarbeitung', vorverarbeitung),
        ('regressor', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
    ])

    modell.fit(X_train, y_train)
    y_vorhersage = modell.predict(X_test)

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

schwelle = 200
y_test_bin = (y_test >= schwelle).astype(int)
y_pred_bin = (y_vorhersage >= schwelle).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test_bin, y_pred_bin, labels=[0, 1]).ravel()

sensitivität = tp / (tp + fn) if (tp + fn) > 0 else np.nan
spezifizität = tn / (tn + fp) if (tn + fp) > 0 else np.nan

matrix_df = pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=['Ist: nicht teuer', 'Ist: teuer'],
    columns=['Prognose: nicht teuer', 'Prognose: teuer']
)

print(f'Bedingung: teuer >= {schwelle} USD')
print('\nWahrheitsmatrix:')
print(matrix_df.to_string())

print(f'\nSensitivität: {sensitivität:.3f}')
print(f'Spezifizität: {spezifizität:.3f}')

Bedingung: teuer >= 200 USD

Wahrheitsmatrix:
                  Prognose: nicht teuer  Prognose: teuer
Ist: nicht teuer                   6726             1104
Ist: teuer                          660             1287

Sensitivität: 0.661
Spezifizität: 0.859


## 4.4 Bewertung mit Hypothesen

Das Modell zeigt brauchbare Ergebnisse, ist aber für präzise Preisprognosen noch zu schwach. Mit MAE 65.58 USD ist der durchschnittliche Fehler akzeptabel, doch RMSE 231.78 USD und R² = 0.112 zeigen grosse Streuung und geringe erklärte Varianz. Auch die Klassifikation ist unausgewogen: Sensitivität 0.661, Spezifität 0.859. Hypothese 1: Wichtige Preistreiber wie Ausstattung, Bildqualität und Beschreibungstext fehlen. Hypothese 2: Sehr teure Inserate sind selten und werden deshalb systematisch unterschätzt. Hypothese 3: Bessere Features und Tuning würden die Leistung erhöhen.